<a href="https://colab.research.google.com/github/SnehaDharne/OcularDiseaseRecognition/blob/main/hyp_resnet50_mirnet_msr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# ResNet50 Multi-Label Ocular Disease Recognition
Screening 8 ophthalmic conditions from retinal fundus photographs.

**Architecture & Pipeline**:
- **Partition**: 80% Training, 10% Validation, 10% Dedicated Holdout Testing (Stratified).
- **Multi-Label Classification**: Sigmoid outputs + Binary Cross-Entropy (8 independent condition detectors).
- **Two-Phase Transfer Learning**:
  - *Phase 1*: Classification Head Warmup (Frozen ResNet50 backbone, learning rate = 3e-4, 8 epochs).
  - *Phase 2*: Deep Fine-Tuning (Unfreeze conv5 block of ResNet50, learning rate = 2e-5 + ReduceLROnPlateau, up to 15 epochs).
- **Dedicated Holdout Evaluation**: Comprehensive accuracy, precision, recall, F1, and AUC reporting on unseen test data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Fast extract directly to Colab local SSD (takes ~5-10 seconds)
!unzip -q "/content/drive/MyDrive/archive (2).zip" -d "/content/data"
print("Extraction complete!")


In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

# Path to the extracted datasets folder containing the 8 disease subfolders
data_dir = "/content/data/datasets"

# 1. Discover all 8 disease classes (exclude hidden folders like .ipynb_checkpoints)
classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)) and not d.startswith('.')])
class_to_idx = {c: i for i, c in enumerate(classes)}
print("Discovered 8 Disease Categories:", class_to_idx)

# 2. Build multi-label DataFrame by scanning the directory (no external CSV required)
records = []
for c in classes:
    class_folder = os.path.join(data_dir, c)
    for fname in os.listdir(class_folder):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(class_folder, fname)
            records.append({
                'filepath': full_path,
                'label': c,
                'class_idx': class_to_idx[c]
            })

df = pd.DataFrame(records)

# Create 8 binary multi-hot target indicator columns
for c in classes:
    df[c] = (df['label'] == c).astype(np.float32)

print(f"Total dataset: {len(df)} images across {len(classes)} classes.")

# 3. Stratified 80% Train, 10% Validation, 10% Test Split
train_df, temp_df = train_test_split(
    df, 
    test_size=0.20, 
    random_state=42, 
    stratify=df['label']
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.50, 
    random_state=42, 
    stratify=temp_df['label']
)

print("\n" + "=" * 55)
print(" DATA SPLIT SUMMARY (80% / 10% / 10% Stratified)")
print("=" * 55)
print(f" - Training Set   : {len(train_df):5d} images ({len(train_df)/len(df)*100:5.1f}%)")
print(f" - Validation Set : {len(val_df):5d} images ({len(val_df)/len(df)*100:5.1f}%)")
print(f" - Test Set       : {len(test_df):5d} images ({len(test_df)/len(df)*100:5.1f}%)")
print("=" * 55)

# 4. Multi-Label Image Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True
)

eval_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col=classes,
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw',
    shuffle=True
)

val_generator = eval_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col=classes,
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw',
    shuffle=False
)

test_generator = eval_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col=classes,
    target_size=(224, 224),
    batch_size=32,
    class_mode='raw',
    shuffle=False
)


In [ ]:
import numpy as np

# Multi-label class distribution analysis on the 80% training split
print("Multi-Label Class Distribution (80% Training Split):")
print("-" * 65)
for idx, cname in enumerate(classes):
    count = int(np.sum(train_df['class_idx'] == idx))
    pct = count / len(train_df) * 100
    print(f" - {cname:<20} : {count:4d} images ({pct:5.2f}%)")
print("-" * 65)


In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC, BinaryAccuracy
import tensorflow as tf

# Base ResNet50 feature extractor
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Frozen for Phase 1 Warmup

# Multi-Label Classification Head:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dense(len(classes), activation='sigmoid')  # 8 Sigmoid outputs for independent probabilities
])

# Phase 1 compilation with num_labels explicitly set for multi_label AUC:
model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=[
        BinaryAccuracy(name='accuracy'),
        AUC(name='auc', multi_label=True, num_labels=len(classes))
    ]
)

print(f"Model built with {len(classes)} multi-label sigmoid outputs.")
model.summary()


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ==============================================================================
# PHASE 1: CLASSIFICATION HEAD WARMUP (FROZEN BACKBONE)
# ==============================================================================
save_path = "/content/drive/MyDrive/best_resnet50_multilabel.keras"

checkpoint_p1 = ModelCheckpoint(
    filepath=save_path,
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    verbose=1
)

early_stop_p1 = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=3,
    restore_best_weights=True
)

print("Starting Phase 1: Training classification head (8 epochs)...")
history_p1 = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=8,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=[checkpoint_p1, early_stop_p1]
)
print("Phase 1 Warmup Complete! Best val_auc recorded.")


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC, BinaryAccuracy
import tensorflow as tf

# ==============================================================================
# PHASE 2: DEEP FINE-TUNING (UNFREEZING CONV5 BLOCK OF RESNET50)
# ==============================================================================
print("Unfreezing top 30 layers (conv5 block) of ResNet50 for ocular feature adaptation...")
base_model.trainable = True

# Freeze all lower layers, keep only the top 30 layers (conv5_block1 to conv5_block3) trainable
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f"Trainable parameters for Fine-Tuning: {trainable_params:,}")

# Recompile with a low learning rate (2e-5) to fine-tune retinal representations
model.compile(
    optimizer=Adam(learning_rate=0.00002),
    loss='binary_crossentropy',
    metrics=[
        BinaryAccuracy(name='accuracy'),
        AUC(name='auc', multi_label=True, num_labels=len(classes))
    ]
)

save_path = "/content/drive/MyDrive/best_resnet50_multilabel.keras"

checkpoint_ft = ModelCheckpoint(
    filepath=save_path,
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    verbose=1
)

early_stop_ft = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=5,
    restore_best_weights=True
)

reduce_lr_ft = ReduceLROnPlateau(
    monitor='val_auc',
    mode='max',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

print("Starting Phase 2 Fine-Tuning (up to 15 epochs with ReduceLROnPlateau)...")
history_fine = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=15,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=[checkpoint_ft, early_stop_ft, reduce_lr_ft]
)
print("Phase 2 Fine-Tuning Complete! Model weights successfully optimized.")


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/best_resnet50_multilabel.keras")


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    hamming_loss
)
import tensorflow as tf

# ==============================================================================
# DEDICATED 10% HOLDOUT TEST EVALUATION WITH CLINICAL THRESHOLD OPTIMIZATION
# ==============================================================================

# 1. Load the Best Fine-Tuned Checkpointed Model
model_eval_path = "/content/drive/MyDrive/best_resnet50_multilabel.keras"
eval_model = tf.keras.models.load_model(model_eval_path)
print(f"Loaded verified model from: {model_eval_path}")

# 2. Predict on Validation Set to Find Optimal F1 Threshold Per Disease
# (Determining thresholds on validation split avoids data leakage)
val_generator.reset()
val_steps = int(np.ceil(len(val_df) / 32))
print(f"\n1. Finding optimal decision thresholds on validation set ({len(val_df)} images)...")
y_val_prob = eval_model.predict(val_generator, steps=val_steps, verbose=0)[:len(val_df)]
y_val_true = np.array(val_df[classes].values, dtype=np.float32)

optimal_thresholds = {}
val_f1_baseline = {}
val_f1_optimal = {}

for i, cname in enumerate(classes):
    best_t = 0.50
    best_f1 = f1_score(y_val_true[:, i], (y_val_prob[:, i] >= 0.50).astype(int), zero_division=0)
    val_f1_baseline[cname] = best_f1
    
    for t in np.arange(0.10, 0.70, 0.02):
        f1 = f1_score(y_val_true[:, i], (y_val_prob[:, i] >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = round(float(t), 2)
            
    optimal_thresholds[cname] = best_t
    val_f1_optimal[cname] = best_f1

print("\nDISCOVERED CLINICAL THRESHOLDS (Optimized on Validation Set):")
print("-" * 65)
print(f"{'DISEASE':<20} | {'DEFAULT T':<10} | {'OPTIMAL T':<10} | {'VAL F1 BOOST'}")
print("-" * 65)
for c in classes:
    print(f"{c:<20} | 0.50       | {optimal_thresholds[c]:<10.2f} | {val_f1_baseline[c]*100:5.1f}% -> {val_f1_optimal[c]*100:5.1f}%")
print("-" * 65)

# 3. Generate Predictions on Completely Unseen 10% Test Split (640 images)
test_generator.reset()
test_steps = int(np.ceil(len(test_df) / 32))
print(f"\n2. Evaluating {len(test_df)} unseen test images across {test_steps} batches...")
y_test_prob = eval_model.predict(test_generator, steps=test_steps, verbose=0)[:len(test_df)]
y_test_true = np.array(test_df[classes].values, dtype=np.float32)

# Multi-label predictions with DEFAULT (0.50) vs OPTIMAL thresholds
y_pred_default = (y_test_prob >= 0.50).astype(int)

y_pred_calibrated = np.zeros_like(y_test_prob, dtype=int)
for i, c in enumerate(classes):
    y_pred_calibrated[:, i] = (y_test_prob[:, i] >= optimal_thresholds[c]).astype(int)

# 4. Compute Aggregate Metrics Comparison
def compute_metrics(y_true, y_pred, y_prob):
    return {
        'mean_acc': (1.0 - hamming_loss(y_true, y_pred)) * 100,
        'exact_acc': accuracy_score(y_true, y_pred) * 100,
        'macro_p': precision_score(y_true, y_pred, average='macro', zero_division=0) * 100,
        'macro_r': recall_score(y_true, y_pred, average='macro', zero_division=0) * 100,
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0) * 100,
        'macro_auc': roc_auc_score(y_true, y_prob, average='macro') * 100,
        'micro_p': precision_score(y_true, y_pred, average='micro', zero_division=0) * 100,
        'micro_r': recall_score(y_true, y_pred, average='micro', zero_division=0) * 100,
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0) * 100,
        'micro_auc': roc_auc_score(y_true, y_prob, average='micro') * 100
    }

m_def = compute_metrics(y_test_true, y_pred_default, y_test_prob)
m_cal = compute_metrics(y_test_true, y_pred_calibrated, y_test_prob)

print("\n" + "=" * 80)
print("     OVERALL HOLDOUT TEST PERFORMANCE: DEFAULT (0.50) vs CALIBRATED")
print("=" * 80)
print(f"{'METRIC':<32} | {'DEFAULT (0.50)':<16} | {'CALIBRATED':<16} | {'IMPROVEMENT'}")
print("-" * 80)
print(f"{'Macro-Averaged Recall':<32} | {m_def['macro_r']:6.2f}%          | {m_cal['macro_r']:6.2f}%          | {m_cal['macro_r'] - m_def['macro_r']:+6.2f}%")
print(f"{'Macro-Averaged F1-Score':<32} | {m_def['macro_f1']:6.2f}%          | {m_cal['macro_f1']:6.2f}%          | {m_cal['macro_f1'] - m_def['macro_f1']:+6.2f}%")
print(f"{'Macro-Averaged Precision':<32} | {m_def['macro_p']:6.2f}%          | {m_cal['macro_p']:6.2f}%          | {m_cal['macro_p'] - m_def['macro_p']:+6.2f}%")
print(f"{'Micro-Averaged Recall':<32} | {m_def['micro_r']:6.2f}%          | {m_cal['micro_r']:6.2f}%          | {m_cal['micro_r'] - m_def['micro_r']:+6.2f}%")
print(f"{'Micro-Averaged F1-Score':<32} | {m_def['micro_f1']:6.2f}%          | {m_cal['micro_f1']:6.2f}%          | {m_cal['micro_f1'] - m_def['micro_f1']:+6.2f}%")
print(f"{'Mean Binary Accuracy (Hamming)':<32} | {m_def['mean_acc']:6.2f}%          | {m_cal['mean_acc']:6.2f}%          | {m_cal['mean_acc'] - m_def['mean_acc']:+6.2f}%")
print(f"{'Macro ROC-AUC':<32} | {m_def['macro_auc']:6.2f}%          | {m_cal['macro_auc']:6.2f}%          | {'Rank Invariant'}")
print(f"{'Micro ROC-AUC':<32} | {m_def['micro_auc']:6.2f}%          | {m_cal['micro_auc']:6.2f}%          | {'Rank Invariant'}")
print("=" * 80)

# 5. Granular Per-Disease Calibrated Breakdown Table
print("\n" + "-" * 95)
print(f"{'DISEASE CATEGORY':<18} | {'THRESH':<6} | {'SUPPORT':<7} | {'ACCURACY':<9} | {'PRECISION':<9} | {'RECALL':<8} | {'F1-SCORE':<8} | {'ROC-AUC':<8}")
print("-" * 95)

for i, cname in enumerate(classes):
    c_true = y_test_true[:, i]
    c_pred = y_pred_calibrated[:, i]
    c_prob = y_test_prob[:, i]
    t = optimal_thresholds[cname]
    
    support = int(np.sum(c_true))
    c_acc   = accuracy_score(c_true, c_pred)
    c_prec  = precision_score(c_true, c_pred, zero_division=0)
    c_rec   = recall_score(c_true, c_pred, zero_division=0)
    c_f1    = f1_score(c_true, c_pred, zero_division=0)
    try:
        c_auc = roc_auc_score(c_true, c_prob)
    except ValueError:
        c_auc = 0.5
    
    print(f"{cname:<18} | {t:<6.2f} | {support:6d}  | {c_acc*100:7.2f}% | {c_prec*100:7.2f}% | {c_rec*100:6.2f}% | {c_f1*100:6.2f}% | {c_auc*100:6.2f}%")

print("-" * 95)

# 6. Detailed Scikit-Learn Classification Report on Calibrated Predictions
print("\nCALIBRATED CLASSIFICATION REPORT (UNSEEN 10% TEST SET):")
print(classification_report(y_test_true, y_pred_calibrated, target_names=classes, digits=4, zero_division=0))

# 7. Sample Multi-Label Inference Comparison
print("=" * 80)
print("SAMPLE MULTI-LABEL INFERENCE (Ground Truth vs Calibrated Predictions):")
print("=" * 80)
for idx in range(min(5, len(test_df))):
    actual = [classes[j] for j in range(8) if y_test_true[idx, j] == 1]
    pred_def = [classes[j] for j in range(8) if y_pred_default[idx, j] == 1]
    pred_cal = [classes[j] for j in range(8) if y_pred_calibrated[idx, j] == 1]
    
    print(f"\n[Test Image #{idx+1}]: {os.path.basename(test_df.iloc[idx]['filepath'])}")
    print(f"  Ground Truth:         {actual}")
    print(f"  Default (0.50):       {pred_def if pred_def else ['None detected']}")
    print(f"  Calibrated (Optimal): {pred_cal if pred_cal else ['None detected']}")
    print("  Probability Breakdown vs Threshold:")
    for j, cname in enumerate(classes):
        prob = y_test_prob[idx, j] * 100
        thresh = optimal_thresholds[cname] * 100
        status = " [DETECTED]" if prob >= thresh else ""
        bar = "#" * int(prob / 5)
        print(f"    - {cname:<18}: {prob:5.2f}% (thresh: {thresh:4.1f}%){status:<12} {bar}")
print("=" * 80)


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc

# ==============================================================================
# MULTI-LABEL ROC CURVES EVALUATION (10% HOLDOUT TEST SET)
# ==============================================================================

# 1. Ensure test predictions and ground truth are loaded
if 'y_test_true' not in globals() or 'y_test_prob' not in globals():
    import tensorflow as tf
    model_eval_path = "/content/drive/MyDrive/best_resnet50_multilabel.keras"
    eval_model = tf.keras.models.load_model(model_eval_path)
    test_generator.reset()
    test_steps = int(np.ceil(len(test_df) / 32))
    y_test_prob = eval_model.predict(test_generator, steps=test_steps, verbose=0)[:len(test_df)]
    y_test_true = np.array(test_df[classes].values, dtype=np.float32)

n_classes = len(classes)

# 2. Compute Per-Class False Positive Rate (FPR) & True Positive Rate (TPR)
fpr = dict()
tpr = dict()
roc_auc = dict()

for i, cname in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_true[:, i], y_test_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# 3. Compute Micro-Average ROC Curve (Aggregated globally across all labels)
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_true.ravel(), y_test_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# 4. Compute Macro-Average ROC Curve (Average of per-class interpolated TPR)
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# 5. Plot Publication-Quality ROC Curves
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, ax = plt.subplots(figsize=(10.5, 8.5), dpi=150)

# Micro & Macro Average Curves
ax.plot(
    fpr["micro"], tpr["micro"],
    label=f"Micro-Average (AUC = {roc_auc['micro']*100:.2f}%)",
    color="deeppink", linestyle=":", linewidth=3.2
)

ax.plot(
    fpr["macro"], tpr["macro"],
    label=f"Macro-Average (AUC = {roc_auc['macro']*100:.2f}%)",
    color="navy", linestyle=":", linewidth=3.2
)

# Professional color palette for 8 ophthalmic categories
palette = [
    "#e41a1c",  # ageDegeneration
    "#377eb8",  # cataract
    "#4daf4a",  # diabetes
    "#984ea3",  # glaucoma
    "#ff7f00",  # hypertension
    "#a65628",  # myopia
    "#008080",  # normal (teal)
    "#7f7f7f"   # others (slate gray)
]

# Sort diseases by AUC descending for clean legend order
sorted_indices = sorted(range(n_classes), key=lambda k: roc_auc[k], reverse=True)

for i in sorted_indices:
    cname = classes[i]
    ax.plot(
        fpr[i], tpr[i],
        color=palette[i % len(palette)],
        linewidth=2.2,
        label=f"{cname:<16} (AUC = {roc_auc[i]*100:.2f}%)"
    )

# Baseline chance diagonal
ax.plot([0, 1], [0, 1], "k--", lw=1.5, alpha=0.7, label="Chance Baseline (AUC = 50.00%)")

# Styling & Labels
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.05])
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel("True Positive Rate (Sensitivity / Recall)", fontsize=13, fontweight='bold', labelpad=10)
ax.set_title("Multi-Label ROC Curves on 10% Holdout Test Set\nOcular Disease Recognition (ResNet50 Transfer Learning)",
             fontsize=14, fontweight='bold', pad=15)
ax.tick_params(axis='both', which='major', labelsize=11)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="lower right", fontsize=10, frameon=True, framealpha=0.95, facecolor="white", edgecolor="#cccccc")

plt.tight_layout()

# 6. Save High-Res PNG to Google Drive for Documentation/Report
save_fig_path = "/content/drive/MyDrive/roc_curves_multilabel.png"
plt.savefig(save_fig_path, dpi=300, bbox_inches='tight')
print(f"\nHigh-resolution ROC curves successfully saved to: {save_fig_path}")

plt.show()
